# Notebook 06 – Déploiement du modèle

## Objectif

Après avoir entraîné et optimisé le meilleur modèle Random Forest dans le Notebook 05, ce notebook montre comment le charger depuis le fichier `best_model.pkl` et l'utiliser pour prédire le risque de sinistre d'un nouveau contrat d'assurance automobile.

Cette étape constitue la transition entre la partie Data Science et la future application Django.

In [1]:
import pandas as pd
import joblib

pd.set_option("display.max_columns", None)

In [2]:
model = joblib.load("../models/best_model.pkl")

print("✅ Modèle chargé avec succès.")

✅ Modèle chargé avec succès.


In [3]:
new_client = pd.DataFrame([{
    "Exposure": 1.0,
    "VehPower": 7,
    "VehAge": 8,
    "DrivAge": 24,
    "BonusMalus": 147,
    "VehBrand": "B12",
    "VehGas": "Regular",
    "Area": "D",
    "Density": 1541,
    "Region": "Centre"
}])

new_client

,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1.0,7,8,24,147,B12,Regular,D,1541,Centre


In [4]:
prediction = model.predict(new_client)[0]
probability = model.predict_proba(new_client)[0]

print("Classe prédite :", prediction)
print(f"Probabilité No Claim : {probability[0]:.2%}")
print(f"Probabilité Claim : {probability[1]:.2%}")

Classe prédite : 1
Probabilité No Claim : 23.88%
Probabilité Claim : 76.12%


In [5]:
if prediction == 1:
    print("🚨 Risque élevé : le modèle prédit un SINISTRE.")
else:
    print("✅ Risque faible : le modèle prédit AUCUN SINISTRE.")

🚨 Risque élevé : le modèle prédit un SINISTRE.


In [6]:
risk = probability[1]

if risk < 0.30:
    level = "Faible"
elif risk < 0.60:
    level = "Moyen"
else:
    level = "Élevé"

print(f"Niveau de risque : {level}")
print(f"Probabilité estimée de sinistre : {risk:.2%}")

Niveau de risque : Élevé
Probabilité estimée de sinistre : 76.12%


In [7]:
clients = pd.DataFrame([
    {
        "Exposure": 1.0,
        "VehPower": 7,
        "VehAge": 8,
        "DrivAge": 24,
        "BonusMalus": 147,
        "VehBrand": "B12",
        "VehGas": "Regular",
        "Area": "D",
        "Density": 1541,
        "Region": "Centre"
    },
    {
        "Exposure": 0.4,
        "VehPower": 5,
        "VehAge": 2,
        "DrivAge": 45,
        "BonusMalus": 50,
        "VehBrand": "B1",
        "VehGas": "Diesel",
        "Area": "B",
        "Density": 120,
        "Region": "Bretagne"
    }
])

predictions = model.predict(clients)
probabilities = model.predict_proba(clients)

results = clients.copy()
results["Prediction"] = ["Claim" if p == 1 else "No Claim" for p in predictions]
results["Probabilité_Claim"] = (probabilities[:,1]*100).round(2)

results

,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region,Prediction,Probabilité_Claim
0,1.0,7,8,24,147,B12,Regular,D,1541,Centre,Claim,76.12
1,0.4,5,2,45,50,B1,Diesel,B,120,Bretagne,No Claim,44.88


In [8]:
results["Niveau_Risque"] = results["Probabilité_Claim"].apply(
    lambda x: "Élevé" if x >= 60 else ("Moyen" if x >= 30 else "Faible")
)

results

,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region,Prediction,Probabilité_Claim,Niveau_Risque
0,1.0,7,8,24,147,B12,Regular,D,1541,Centre,Claim,76.12,Élevé
1,0.4,5,2,45,50,B1,Diesel,B,120,Bretagne,No Claim,44.88,Moyen
